# release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1

Full constrained actor probe review for the release16 rulebase-reversal drawdown/upside state-head profile.

The training command was:

```bash
PYTHONPATH=. /Users/rziiiii/dev/workspace/qlib_venv/bin/python -m src.soxl_quant.cli \
  --profiles release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1 \
  --walk-forward --skip-data-refresh --seed 20260615
```

This notebook is the durable review surface for the run. It reads the fixed-split summary, fold-local walk-forward worker JSONs, and official daily CSV, then evaluates the current hard gates: MaxDD below 20%, walk-forward quality, SOXLBacktest qlib YTD reference alignment, current-best YTD alignment, promotion, and actor retention.


In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PROFILE = "release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1"
ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
CARD = ROOT / "runtime" / "reward" / "card" / PROFILE
MODEL_CARD = ROOT / "runtime" / "model" / "card" / PROFILE
WF_DIR = ROOT / "runtime" / "soxl" / "wf_workers"
AUDIT_SUMMARY = ROOT / "runtime" / "model_selection" / "card" / "retraining_retention_current_audit_v1" / "summary.json"
HANDOFF_SUMMARY = ROOT / "runtime" / "model_selection" / "card" / "retraining_execution_handoff_current_v1" / "summary.json"

SUMMARY_PATH = MODEL_CARD / f"soxl_{PROFILE}_summary.json"
DAILY_PATH = MODEL_CARD / f"{PROFILE}_soxl_daily.csv"
WF_PATHS = {
    "wf_2020": WF_DIR / f"{PROFILE}-wf_2020-seed20260615.json",
    "wf_2022": WF_DIR / f"{PROFILE}-wf_2022-seed20260615.json",
    "wf_2023_2024": WF_DIR / f"{PROFILE}-wf_2023_2024-seed20260615.json",
}

def read_json(path: Path) -> dict:
    with path.open() as fh:
        return json.load(fh)

summary = read_json(SUMMARY_PATH)
audit = read_json(AUDIT_SUMMARY)
handoff = read_json(HANDOFF_SUMMARY)
wf = {name: read_json(path) for name, path in WF_PATHS.items()}

daily = pd.read_csv(DAILY_PATH, parse_dates=["datetime"])
ytd = daily[daily["datetime"].dt.year == 2026].copy()
assert not ytd.empty, "daily replay must contain 2026 rows for the YTD gate"

ytd_return = (ytd["strategy_nav"].iloc[-1] / ytd["strategy_nav"].iloc[0] - 1.0) * 100.0
ytd_buyhold_return = (ytd["buyhold_nav"].iloc[-1] / ytd["buyhold_nav"].iloc[0] - 1.0) * 100.0
ytd_drawdown = ((ytd["strategy_nav"] / ytd["strategy_nav"].cummax()) - 1.0).min() * 100.0

retention_gate = audit["current_retention_gate"]
reference_return = float(retention_gate["reference_return_pct"])
current_best_floor = float(retention_gate["required_best_return_pct"])
max_wf_drawdown = float(retention_gate["max_walkforward_drawdown_pct"])
min_wf_relative = 30.0

fixed = summary["test_proxy"]
metrics_rows = [
    {
        "window": "fixed_test_2020_2026",
        "return_pct": fixed["return"],
        "sharpe": fixed["sharpe"],
        "max_drawdown_pct": fixed["max_dd"],
        "relative_return_pp": fixed.get("relative_return"),
        "trades": None,
    },
    {
        "window": "ytd_2026_daily_replay",
        "return_pct": ytd_return,
        "sharpe": None,
        "max_drawdown_pct": abs(ytd_drawdown),
        "relative_return_pp": ytd_return - ytd_buyhold_return,
        "trades": None,
    },
]
for name, result in wf.items():
    metrics_rows.append(
        {
            "window": name,
            "return_pct": result["return"],
            "sharpe": result["sharpe"],
            "max_drawdown_pct": result["max_dd"],
            "relative_return_pp": result.get("relative_return"),
            "trades": result.get("trades"),
        }
    )
metrics = pd.DataFrame(metrics_rows)
metrics


,window,return_pct,sharpe,max_drawdown_pct,relative_return_pp,trades
0,fixed_test_2020_2026,291.034886,1.054521,25.329680,-2740.481805,NaN
1,ytd_2026_daily_replay,38.555494,NaN,9.559553,-288.370809,NaN
2,wf_2020,35.374744,1.278224,19.260466,-297.958603,33.0
3,wf_2022,-16.467158,-1.079970,18.149535,62.105998,28.0
4,wf_2023_2024,-7.690929,-0.308488,20.914378,28.500773,34.0


In [2]:
def gate(name: str, actual, op: str, limit, passed: bool, scope: str, note: str) -> dict:
    return {
        "gate": name,
        "scope": scope,
        "actual": actual,
        "operator": op,
        "limit": limit,
        "passed": bool(passed),
        "note": note,
    }

gates = []
gates.append(gate("fixed_test_max_drawdown", fixed["max_dd"], "<", 20.0, fixed["max_dd"] < 20.0, "fixed_test_2020_2026", "User hard drawdown gate."))
gates.append(gate("fixed_promotion_eligible", summary["promotion"]["eligible"], "==", True, summary["promotion"]["eligible"] is True, "fixed_plus_wf", "Repo promotion bundle must pass."))
gates.append(gate("actor_retention_allowed", summary["actor_retention"]["final_retention_allowed"], "==", True, summary["actor_retention"]["final_retention_allowed"] is True, "retention", "No actor may be retained if strict retention rejects it."))
gates.append(gate("ytd_return_vs_soxlbacktest_reference", ytd_return, ">=", reference_return, ytd_return >= reference_return, "ytd_2026_daily_replay", "SOXLBacktest qlib reference baseline."))
gates.append(gate("ytd_return_vs_current_best_floor", ytd_return, ">=", current_best_floor, ytd_return >= current_best_floor, "ytd_2026_daily_replay", "Current-best alignment floor from retention audit."))
gates.append(gate("ytd_max_drawdown", abs(ytd_drawdown), "<", 20.0, abs(ytd_drawdown) < 20.0, "ytd_2026_daily_replay", "YTD drawdown sanity check."))
for name, result in wf.items():
    gates.append(gate(f"{name}_return_nonnegative", result["return"], ">=", 0.0, result["return"] >= 0.0, name, "Fold return must not be negative."))
    gates.append(gate(f"{name}_max_drawdown", result["max_dd"], "<", max_wf_drawdown, result["max_dd"] < max_wf_drawdown, name, "Walk-forward drawdown hard gate."))
    gates.append(gate(f"{name}_relative_outperformance", result.get("relative_return"), ">=", min_wf_relative, result.get("relative_return", float("-inf")) >= min_wf_relative, name, "Relative return floor from handoff validation list."))
    gates.append(gate(f"{name}_sharpe_nonnegative", result["sharpe"], ">=", 0.0, result["sharpe"] >= 0.0, name, "Fold Sharpe must not be negative."))

gates_df = pd.DataFrame(gates)
failed_gates = gates_df.loc[~gates_df["passed"], "gate"].tolist()
decision = "full_probe_failed_no_actor_retention" if failed_gates else "full_probe_passed_candidate_for_retention_review"
promotion_allowed = False if failed_gates else bool(summary["promotion"]["eligible"])
actor_retention_allowed = False if failed_gates else bool(summary["actor_retention"]["final_retention_allowed"])

gates_df


,gate,scope,actual,operator,limit,passed,note
0,fixed_test_max_drawdown,fixed_test_2020_2026,25.32968,<,20.0,False,User hard drawdown gate.
1,fixed_promotion_eligible,fixed_plus_wf,False,==,True,False,Repo promotion bundle must pass.
2,actor_retention_allowed,retention,False,==,True,False,No actor may be retained if strict retention r...
3,ytd_return_vs_soxlbacktest_reference,ytd_2026_daily_replay,38.555494,>=,132.077978,False,SOXLBacktest qlib reference baseline.
4,ytd_return_vs_current_best_floor,ytd_2026_daily_replay,38.555494,>=,213.143462,False,Current-best alignment floor from retention au...
5,ytd_max_drawdown,ytd_2026_daily_replay,9.559553,<,20.0,True,YTD drawdown sanity check.
6,wf_2020_return_nonnegative,wf_2020,35.374744,>=,0.0,True,Fold return must not be negative.
7,wf_2020_max_drawdown,wf_2020,19.260466,<,20.0,True,Walk-forward drawdown hard gate.
8,wf_2020_relative_outperformance,wf_2020,-297.958603,>=,30.0,False,Relative return floor from handoff validation ...
9,wf_2020_sharpe_nonnegative,wf_2020,1.278224,>=,0.0,True,Fold Sharpe must not be negative.


In [3]:
CARD.mkdir(parents=True, exist_ok=True)
summary_out = {
    "experiment": PROFILE,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "promotion_allowed": promotion_allowed,
    "actor_retention_allowed": actor_retention_allowed,
    "seed": 20260615,
    "command": "PYTHONPATH=. /Users/rziiiii/dev/workspace/qlib_venv/bin/python -m src.soxl_quant.cli --profiles release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1 --walk-forward --skip-data-refresh --seed 20260615",
    "fixed_test": {
        "return_pct": fixed["return"],
        "sharpe": fixed["sharpe"],
        "max_drawdown_pct": fixed["max_dd"],
        "relative_return_pp": fixed.get("relative_return"),
    },
    "ytd_2026_daily_replay": {
        "start": str(ytd["datetime"].iloc[0].date()),
        "end": str(ytd["datetime"].iloc[-1].date()),
        "rows": int(len(ytd)),
        "return_pct": ytd_return,
        "buyhold_return_pct": ytd_buyhold_return,
        "relative_return_pp": ytd_return - ytd_buyhold_return,
        "max_drawdown_pct": abs(ytd_drawdown),
        "soxlbacktest_reference_return_pct": reference_return,
        "current_best_required_return_pct": current_best_floor,
    },
    "walk_forward": {
        name: {
            "return_pct": result["return"],
            "sharpe": result["sharpe"],
            "max_drawdown_pct": result["max_dd"],
            "relative_return_pp": result.get("relative_return"),
            "trades": result.get("trades"),
        }
        for name, result in wf.items()
    },
    "failed_gates": failed_gates,
    "sidecars": [
        "full_probe_gate_summary.csv",
        "summary.json",
        "conclusion.md",
    ],
    "evidence_inputs": {
        "fixed_summary": str(SUMMARY_PATH.relative_to(ROOT)),
        "daily_replay": str(DAILY_PATH.relative_to(ROOT)),
        "wf_workers": {name: str(path.relative_to(ROOT)) for name, path in WF_PATHS.items()},
    },
    "checkpoint_retention": {
        "generated_actor_paths_removed": True,
        "reason": "promotion blocked and actor_retention.final_retention_allowed=false",
    },
}

metrics.to_csv(CARD / "full_probe_metrics.csv", index=False)
gates_df.to_csv(CARD / "full_probe_gate_summary.csv", index=False)
(CARD / "summary.json").write_text(json.dumps(summary_out, indent=2) + "\n")

conclusion = f"""# {PROFILE} full probe conclusion\n\nDecision: `{decision}`.\n\nThe constrained full walk-forward probe is not retainable. Fixed test return is {fixed['return']:.1f}% with MaxDD {fixed['max_dd']:.1f}%, which violates the user hard MaxDD <20% gate. The 2026 YTD daily replay returns {ytd_return:.1f}% from {ytd['datetime'].iloc[0].date()} to {ytd['datetime'].iloc[-1].date()}, below both the SOXLBacktest qlib reference {reference_return:.1f}% and current-best alignment floor {current_best_floor:.1f}%. Walk-forward remains unstable: wf_2020 return {wf['wf_2020']['return']:.1f}% / MaxDD {wf['wf_2020']['max_dd']:.1f}% / relative {wf['wf_2020'].get('relative_return'):.1f}pp; wf_2022 return {wf['wf_2022']['return']:.1f}% / Sharpe {wf['wf_2022']['sharpe']:.2f}; wf_2023_2024 return {wf['wf_2023_2024']['return']:.1f}% / MaxDD {wf['wf_2023_2024']['max_dd']:.1f}% / relative {wf['wf_2023_2024'].get('relative_return'):.1f}pp.\n\nActor retention is blocked. The generated actor checkpoints are deleted and must not be used as dashboard runnable models. Keep only JSON/CSV evidence for diagnosis.\n\nNext action: do not rerun this profile unchanged. Diagnose the state-head/reward mismatch that causes underparticipation versus SOXLBacktest in 2026 YTD and the negative wf_2022 / wf_2023_2024 folds before opening another retained actor branch.\n"""
(CARD / "conclusion.md").write_text(conclusion)

summary_out


{'experiment': 'release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1',
 'created_at': '2026-06-14T18:21:26.571537+00:00',
 'decision': 'full_probe_failed_no_actor_retention',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'seed': 20260615,
 'command': 'PYTHONPATH=. /Users/rziiiii/dev/workspace/qlib_venv/bin/python -m src.soxl_quant.cli --profiles release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1 --walk-forward --skip-data-refresh --seed 20260615',
 'fixed_test': {'return_pct': 291.03488641953004,
  'sharpe': 1.0545205667232205,
  'max_drawdown_pct': 25.329679974531977,
  'relative_return_pp': -2740.4818046469063},
 'ytd_2026_daily_replay': {'start': '2026-01-02',
  'end': '2026-06-09',
  'rows': 109,
  'return_pct': np.float64(38.555494067096085),
  'buyhold_return_pct': np.float64(326.92630294234766),
  'relative_return_pp': np.float64(-288.37080887525156),
  'max_drawdown_pct': np.float64(9.559552575516284),
  'soxlbacktest_refer

## Conclusion

Decision: `full_probe_failed_no_actor_retention`.

The constrained full walk-forward probe is not retainable. It fails the user hard MaxDD gate on the fixed test replay, fails 2026 YTD return alignment versus both the SOXLBacktest qlib reference and current-best floor, and remains unstable across walk-forward: `wf_2022` is negative, `wf_2023_2024` is negative and above the 20% MaxDD ceiling, and `wf_2020` badly underperforms buy-and-hold on relative return.

No actor checkpoint is retained from this run. The generated actor files are discarded; the JSON/CSV metrics and this notebook remain as diagnostic evidence only.
